# FlowGuard - Phase 5: Hardening
Domain adversarial training + EWC + PGD adversarial training.

In [ ]:
import os, sys

REPO_DIR = "/content/flowguard"
if not os.path.exists(REPO_DIR):
    raise RuntimeError("Run notebook 00_setup_and_validate.ipynb first.")
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

import torch
from src.utils.config import load_config, get_device
from src.utils.reproducibility import setup_reproducibility
from src.models.flowguard import FlowGuard
from src.data.dataset import create_dataloader

config = load_config("configs/phase5_hardening.yaml")
setup_reproducibility(config)
device = get_device(config)
print(f"Device: {device}")

## Domain Adversarial Training

In [ ]:
from src.training.adversarial import domain_adversarial_training_step
from torch.cuda.amp import GradScaler

# Load Phase 3 model
model = FlowGuard(config)
ckpt_path = "checkpoints/phase3/final_global.pt"
if os.path.exists(ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.enable_domain_discriminator(num_domains=4)
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = torch.nn.CrossEntropyLoss()
scaler = GradScaler(enabled=device.type=='cuda')

# Train for a few epochs with domain adversarial
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    n = 0
    
    for ds_idx, ds_info in enumerate(config['data']['datasets']):
        name = ds_info['name']
        path = f"data/splits/protocol_a/{name}_train.parquet"
        if not os.path.exists(path):
            continue
        loader = create_dataloader(path, batch_size=512, num_workers=0)
        
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            domain_labels = torch.full((x.size(0),), ds_idx, dtype=torch.long, device=device)
            lambda_grl = min(1.0, epoch / num_epochs)
            
            loss, _ = domain_adversarial_training_step(
                model, x, y, domain_labels, optimizer, criterion, lambda_grl
            )
            total_loss += loss * x.size(0)
            n += x.size(0)
    
    print(f"Epoch {epoch+1}/{num_epochs}: loss={total_loss/max(n,1):.4f}")

torch.save(model.state_dict(), "checkpoints/phase5/hardened_model.pt")
print("Hardened model saved.")

## EWC Regularization

In [ ]:
from src.training.ewc import compute_and_save_fisher, load_ewc_loss, train_with_ewc

# Compute Fisher on current model
train_path = "data/splits/protocol_a/unsw_train.parquet"
if os.path.exists(train_path):
    loader = create_dataloader(train_path, batch_size=512, shuffle=False, num_workers=0)
    fisher_path = "checkpoints/phase5/fisher.pt"
    compute_and_save_fisher(model, loader, fisher_path, device=device)
    print("Fisher matrix computed and saved.")